In [5]:
from google.colab import drive
drive.mount('/content/drive')
# then, for convenience
%cd /content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!nvidia-smi

Fri Feb 13 21:40:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# ALM / PSyGen settings
alm:
  P_min: 0.7          # minimum acceptable privacy score
  lambda_init: 0.0
  mu_init: 0.1
  lambda_lr: 1e-3     # step size for λ
  mu_growth: 1.5      # µ ← mu_growth * µ when violations persist
  alm_weight: 1.0     # how strongly to weight ALM term vs adv loss

  # quality metric weights (sum to 1)
  w_quality:
    D: 0.3   # distribution similarity
    A: 0.1   # anomaly / rare events
    C: 0.3   # inter-feature associations
    V: 0.2   # diversity / coverage
    T: 0.1   # temporal consistency (0 if not temporal data)

  # privacy metric weights (sum to 1)
  w_privacy:
    DCR: 0.4    # distance to closest real
    Qdelta: 0.3 # quantile difference
    I: 0.3      # duplicate score


SyntaxError: invalid syntax (ipython-input-4072824666.py, line 2)

In [6]:
# Colab-ready: generic loader + ALM GAN + save ONE decoded synthetic CSV
!pip install -q pandas scikit-learn tqdm torch

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ------------------
# Config
# ------------------
DATA_PATH = "/content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-main/data/data.csv"
OUT_PATH  = "/content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-outputs/synthetic_data_alm.csv"

BATCH_SIZE = 256
EPOCHS = 5          # increase later
LATENT_DIM = 64
LR_G = 1e-4
LR_D = 1e-4
N_CRITIC = 5
LAMBDA_GP = 10.0

# ALM config
P_MIN = 0.7
ALM_WEIGHT = 1.0
LAMBDA_INIT = 0.0
MU_INIT = 0.1
LAMBDA_LR = 1e-3
MU_GROWTH = 1.5

WQ = dict(D=0.3, A=0.1, C=0.3, V=0.3, T=0.0)
WP = dict(DCR=0.4, Qdelta=0.3, I=0.3)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ------------------
# Data loader (generic)
# ------------------
def load_data_generic(path):
    df = pd.read_csv(path, low_memory=False)

    # clean strings
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
            df[c] = df[c].replace({"?": np.nan, "": np.nan, "nan": np.nan})

    # keep Target aside (optional)
    target_series = None
    if "Target" in df.columns:
        target_series = df["Target"].copy()
        X_df = df.drop(columns=["Target"])
    else:
        X_df = df.copy()

    # coerce numeric-like
    for c in X_df.columns:
        if X_df[c].dtype == "object":
            coerced = pd.to_numeric(X_df[c], errors="coerce")
            if coerced.notna().mean() > 0.8:
                X_df[c] = coerced

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object" or X_df[c].dtype.name == "category"]
    num_cols = [c for c in X_df.columns if c not in cat_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0,
    )

    X = pre.fit_transform(X_df).astype(np.float32)

    # Needed for decoding later
    ohe = pre.named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
    scaler = pre.named_transformers_["num"].named_steps["scaler"] if len(num_cols) else None

    return df, X_df, X, pre, num_cols, cat_cols, scaler, ohe, target_series

# ------------------
# Dataset
# ------------------
class TabularDataset(Dataset):
    def __init__(self, x):
        self.x = x.astype(np.float32)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, idx):
        return self.x[idx]

# ------------------
# Models
# ------------------
class Generator(nn.Module):
    def __init__(self, latent_dim, data_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 256),
            nn.ReLU(True),
            nn.Linear(256, data_dim),
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, data_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x).view(-1)

# ------------------
# WGAN-GP helpers
# ------------------
def gradient_penalty(D, real, fake):
    batch_size = real.size(0)
    alpha = torch.rand(batch_size, 1, device=real.device).expand_as(real)
    interpolated = alpha * real + (1 - alpha) * fake
    interpolated.requires_grad_(True)
    d_interpolated = D(interpolated)
    gradients = torch.autograd.grad(
        outputs=d_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interpolated),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(batch_size, -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gp

def sample_noise(batch_size, latent_dim, device):
    return torch.randn(batch_size, latent_dim, device=device)

# ------------------
# PSyGen-style Q/P proxies + ALM
# ------------------
class QualityPrivacyMetrics:
    def __init__(self, wq, wp, device="cuda"):
        self.wq, self.wp, self.device = wq, wp, device

    def _distribution_similarity(self, real, fake):
        def rbf(x, y, gamma=1.0):
            xx = torch.cdist(x, x, p=2)
            yy = torch.cdist(y, y, p=2)
            xy = torch.cdist(x, y, p=2)
            return torch.exp(-gamma * xx**2).mean() + torch.exp(-gamma * yy**2).mean() - 2*torch.exp(-gamma * xy**2).mean()
        mmd2 = rbf(real, fake)
        return torch.exp(-mmd2).clamp(0, 1)

    def _anomaly_preservation(self, real, fake):
        q1 = torch.quantile(real, 0.25, dim=0)
        q3 = torch.quantile(real, 0.75, dim=0)
        iqr = q3 - q1 + 1e-6
        hi = q3 + 1.5 * iqr
        lo = q1 - 1.5 * iqr
        rare_r = ((real > hi) | (real < lo)).float().mean()
        rare_f = ((fake > hi) | (fake < lo)).float().mean()
        return (1 - torch.abs(rare_r - rare_f)).clamp(0, 1)

    def _association_preservation(self, real, fake):
        def corr(x):
            x = (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + 1e-6)
            return (x.T @ x) / (x.size(0) - 1)
        diff = torch.mean(torch.abs(corr(real) - corr(fake)))
        return torch.exp(-diff).clamp(0, 1)

    def _diversity(self, fake):
        x = fake - fake.mean(0, keepdim=True)
        x = x / (x.norm(p=2, dim=1, keepdim=True) + 1e-6)
        sim = torch.mm(x, x.T)
        mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
        return (1 - sim[mask].mean()).clamp(0, 1)

    def _dcr(self, real, fake):
        dists = torch.cdist(fake, real, p=2)
        return torch.tanh(dists.min(dim=1)[0].mean())

    def _qdelta(self, real, fake):
        qs = torch.tensor([0.1,0.25,0.5,0.75,0.9], device=real.device)
        diff = torch.abs(torch.quantile(real, qs, dim=0) - torch.quantile(fake, qs, dim=0)).mean()
        return diff / (diff + 1.0)

    def _dup(self, real, fake):
        x = torch.cat([real, fake], dim=0)
        d = torch.cdist(x, x, p=2)
        dup = (d < 1e-3).float() - torch.eye(d.size(0), device=d.device)
        return dup.clamp(min=0).mean()

    def __call__(self, real, fake):
        D = self._distribution_similarity(real, fake)
        A = self._anomaly_preservation(real, fake)
        C = self._association_preservation(real, fake)
        V = self._diversity(fake)
        Q = self.wq["D"]*D + self.wq["A"]*A + self.wq["C"]*C + self.wq["V"]*V

        DCR = self._dcr(real, fake)
        Qd  = self._qdelta(real, fake)
        I   = self._dup(real, fake)

        P = self.wp["DCR"]*DCR - self.wp["Qdelta"]*Qd - self.wp["I"]*I
        return Q, P

class ALMController:
    def __init__(self, P_min, lambda_init, mu_init, lambda_lr, mu_growth, device="cuda"):
        self.P_min = torch.tensor(P_min, device=device)
        self.lmbda = torch.tensor(lambda_init, device=device)
        self.mu = torch.tensor(mu_init, device=device)
        self.lambda_lr = lambda_lr
        self.mu_growth = mu_growth
        self.residual_ma = None
        self.device = device

    def compute_loss(self, Q, P):
        residual = self.P_min - P
        violation = torch.relu(residual)
        L = (1 - Q) + self.lmbda * violation + 0.5 * self.mu * violation**2
        return L, residual.detach()

    def update(self, residual_epoch):
        r = torch.tensor(residual_epoch, device=self.device)
        self.residual_ma = r if self.residual_ma is None else (0.9*self.residual_ma + 0.1*r)
        self.lmbda = (self.lmbda + self.lambda_lr * self.residual_ma).clamp(min=0.0)
        if self.residual_ma > 0:
            self.mu = self.mu * self.mu_growth

# ------------------
# Decode helper (argmax for categoricals)
# ------------------
def decode_one_file(fake_encoded, num_cols, cat_cols, scaler, ohe, X_df_template, target_series=None):
    """
    Produces ONE decoded DataFrame with the original columns (best effort):
      - numeric: inverse StandardScaler
      - categorical: argmax per one-hot group -> category
    """
    n_num = len(num_cols)
    num_block = fake_encoded[:, :n_num] if n_num else np.zeros((len(fake_encoded), 0), dtype=np.float32)
    cat_block = fake_encoded[:, n_num:] if len(cat_cols) else np.zeros((len(fake_encoded), 0), dtype=np.float32)

    # numeric inverse scale
    if n_num and scaler is not None:
        num_dec = scaler.inverse_transform(num_block)
        df_num = pd.DataFrame(num_dec, columns=num_cols)
    else:
        df_num = pd.DataFrame(index=range(len(fake_encoded)))

    # categorical decode (argmax)
    if len(cat_cols) and ohe is not None:
        cats = ohe.categories_
        out = {}
        idx = 0
        for i, col in enumerate(cat_cols):
            width = len(cats[i])
            block = cat_block[:, idx:idx+width]
            arg = np.argmax(block, axis=1)
            out[col] = [cats[i][a] for a in arg]
            idx += width
        df_cat = pd.DataFrame(out)
    else:
        df_cat = pd.DataFrame(index=range(len(fake_encoded)))

    decoded = pd.concat([df_num, df_cat], axis=1)

    # ensure same column order as original X_df
    decoded = decoded.reindex(columns=X_df_template.columns)

    # optional: add Target back (sample from real target distribution)
    if target_series is not None:
        # sample targets with replacement from real
        sampled_target = target_series.sample(n=len(decoded), replace=True, random_state=42).reset_index(drop=True)
        decoded["Target"] = sampled_target

    return decoded

# ------------------
# Load data
# ------------------
print(f"Loading data from {DATA_PATH}")
df_raw, X_df, X_real, preprocessor, num_cols, cat_cols, scaler, ohe, target_series = load_data_generic(DATA_PATH)
dim = X_real.shape[1]
print("Final training matrix shape:", X_real.shape)

dataset = TabularDataset(X_real)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# ------------------
# Train
# ------------------
G = Generator(LATENT_DIM, dim).to(DEVICE)
D = Discriminator(dim).to(DEVICE)
opt_G = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.9))
opt_D = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.9))

qp_metrics = QualityPrivacyMetrics(WQ, WP, device=DEVICE)
alm = ALMController(P_MIN, LAMBDA_INIT, MU_INIT, LAMBDA_LR, MU_GROWTH, device=DEVICE)

G.train(); D.train()

for epoch in range(1, EPOCHS + 1):
    d_losses, g_losses, q_scores, p_scores, residuals = [], [], [], [], []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

    for real_batch in pbar:
        real_batch = real_batch.to(DEVICE)

        # Critic steps
        for _ in range(N_CRITIC):
            z = sample_noise(real_batch.size(0), LATENT_DIM, DEVICE)
            fake_batch = G(z).detach()

            d_real = D(real_batch)
            d_fake = D(fake_batch)
            gp = gradient_penalty(D, real_batch, fake_batch)
            d_loss = -(d_real.mean() - d_fake.mean()) + LAMBDA_GP * gp

            opt_D.zero_grad()
            d_loss.backward()
            opt_D.step()

        # Generator step
        z = sample_noise(real_batch.size(0), LATENT_DIM, DEVICE)
        fake_batch = G(z)
        adv_loss = -D(fake_batch).mean()

        Q, P = qp_metrics(real_batch, fake_batch)
        L_AL, residual = alm.compute_loss(Q, P)

        g_loss = adv_loss + ALM_WEIGHT * L_AL

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        d_losses.append(d_loss.item())
        g_losses.append(g_loss.item())
        q_scores.append(Q.item())
        p_scores.append(P.item())
        residuals.append(residual.item())

        pbar.set_postfix(d_loss=f"{np.mean(d_losses):.3f}",
                         g_loss=f"{np.mean(g_losses):.3f}",
                         Q=f"{np.mean(q_scores):.3f}",
                         P=f"{np.mean(p_scores):.3f}")

    mean_res = float(np.mean(residuals))
    alm.update(mean_res)

    print(f"Epoch {epoch:03d} | D={np.mean(d_losses):.3f} | G={np.mean(g_losses):.3f} | "
          f"Q={np.mean(q_scores):.3f} | P={np.mean(p_scores):.3f} | "
          f"res={mean_res:.3f} | lambda={alm.lmbda.item():.3f} | mu={alm.mu.item():.3f}")

print("Training finished.")

# ------------------
# Generate ONE synthetic file (decoded)
# ------------------
G.eval()
with torch.no_grad():
    N_SYN = 1000
    z = sample_noise(N_SYN, LATENT_DIM, DEVICE)
    fake_encoded = G(z).cpu().numpy()

synthetic_df = decode_one_file(
    fake_encoded=fake_encoded,
    num_cols=num_cols,
    cat_cols=cat_cols,
    scaler=scaler,
    ohe=ohe,
    X_df_template=X_df,
    target_series=target_series,
)

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
synthetic_df.to_csv(OUT_PATH, index=False)
print(f"\n✅ One synthetic file saved to: {OUT_PATH}")
print("Synthetic head:")
print(synthetic_df.head())


Using device: cuda
Loading data from /content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-main/data/data.csv
Final training matrix shape: (100000, 27)


Epoch 1/5:   0%|          | 0/390 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 001 | D=-3.394 | G=-0.976 | Q=0.877 | P=0.283 | res=0.417 | lambda=0.000 | mu=0.150


Epoch 002 | D=-1.450 | G=-0.348 | Q=0.954 | P=0.326 | res=0.374 | lambda=0.001 | mu=0.225


Epoch 003 | D=-0.696 | G=-0.107 | Q=0.963 | P=0.329 | res=0.371 | lambda=0.001 | mu=0.338


Epoch 004 | D=-0.652 | G=0.273 | Q=0.966 | P=0.332 | res=0.368 | lambda=0.002 | mu=0.506


Epoch 005 | D=-0.588 | G=1.112 | Q=0.966 | P=0.335 | res=0.365 | lambda=0.002 | mu=0.759
Training finished.

✅ One synthetic file saved to: /content/drive/MyDrive/Synthetic-Data-generation-main/CTAB-GAN-Plus-outputs/synthetic_data_alm.csv
Synthetic head:
    customer_id  customer_gender  customer_age  is_bank_staff  \
0  63825.453125         0.160431     24.964577       0.001690   
1  62304.203125         0.627230     31.794853       0.000162   
2  56952.898438         0.245904     36.647362       0.070282   
3  91360.359375        -0.690547     30.058012      -0.017824   
4  37804.359375         0.710215     31.824268       0.021657   

   years_with_bank  sms_banking_registered  verification_method  \
0        25.370691                0.691268             0.698203   
1         7.792767                0.819064             1.043304   
2        34.994068                0.682343             0.207691   
3         6.386877                0.368473             0.763619   
4        21.745317 

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from scipy.stats import ks_2samp, entropy
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import spearmanr

# ---------- Q scorer (uses your DACV) ----------
def score_quality_Q(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col="Target"):
    cols = [c for c in real_df.columns if c != target_col]
    cat_cols = [c for c in cols if real_df[c].dtype == "object"]
    num_cols = [c for c in cols if c not in cat_cols]

    # D (numeric hist/KS/JSD + RC; categorical TVD)
    def js_divergence(p, q, base=np.e):
        p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
        p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
        m = 0.5*(p+q);
        return 0.5*entropy(p, m, base=base)+0.5*entropy(q, m, base=base)
    def hist_compare(xr, xs, bins=30, rng=None, smooth=1e-9):
        if rng is None:
            lo = np.nanmin(np.concatenate([xr, xs])); hi = np.nanmax(np.concatenate([xr, xs]));
            if lo==hi: hi = lo+1.0; rng=(lo,hi)
        hr, edges = np.histogram(xr, bins=bins, range=rng)
        hs, _     = np.histogram(xs, bins=bins, range=rng)
        pr = (hr + smooth) / (hr.sum() + smooth*bins)
        ps = (hs + smooth) / (hs.sum() + smooth*bins)
        return {"jsd": float(js_divergence(pr, ps))}
    def tvd_from_counts(p, q):
        keys = sorted(set(p).union(q), key=str)
        pv = np.array([p.get(k, 0.0) for k in keys], dtype=float)
        qv = np.array([q.get(k, 0.0) for k in keys], dtype=float)
        return 0.5 * np.abs(pv - qv).sum()

    D_parts = []
    for c in num_cols:
        xr = real_df[c].dropna().values; xs = syn_df[c].dropna().values
        if xr.size and xs.size:
            ks = ks_2samp(xr, xs).statistic
            jsd = hist_compare(xr, xs)["jsd"]
            lo, hi = np.nanmin(xr), np.nanmax(xr)
            rc = float(np.mean((xs>=lo)&(xs<=hi)))
            D_parts.append( (1-ks, 1-jsd, rc) )
    D_score = np.mean([np.mean(p) for p in D_parts]) if D_parts else np.nan

    real_cat = real_df[cat_cols].copy().astype(str)
    syn_cat  = syn_df[cat_cols].copy().astype(str)
    tvds=[]
    for c in cat_cols:
        pr = real_cat[c].value_counts(normalize=True).to_dict()
        ps = syn_cat[c].value_counts(normalize=True).to_dict()
        tvds.append(1.0 - tvd_from_counts(pr, ps))
    D_cat = float(np.mean(tvds)) if tvds else np.nan

    # C (assoc)
    def spearman_matrix(df, cols):
        m=len(cols); R=np.ones((m,m))*np.nan
        for i,a in enumerate(cols):
            for j,b in enumerate(cols):
                if i==j: R[i,i]=1.0
                elif i<j:
                    rho,_=spearmanr(df[a], df[b], nan_policy="omit")
                    R[i,j]=R[j,i]=rho
        return R
    C_num=np.nan
    if len(num_cols)>=2:
        Rr=spearman_matrix(real_df,num_cols); Rs=spearman_matrix(syn_df,num_cols)
        mask = np.isfinite(Rr)&np.isfinite(Rs)
        C_num = float(1.0 - np.nanmean(np.abs(Rr[mask]-Rs[mask])))

    def nmi_matrix(df, cols):
        m=len(cols); M=np.ones((m,m))*np.nan
        for i,a in enumerate(cols):
            for j,b in enumerate(cols):
                if i==j: M[i,i]=1.0
                elif i<j: M[i,j]=M[j,i]=normalized_mutual_info_score(df[a].astype(str), df[b].astype(str))
        return M
    C_cat=np.nan
    if len(cat_cols)>=2:
        Mr=nmi_matrix(real_cat,cat_cols); Ms=nmi_matrix(syn_cat,cat_cols)
        mask=np.isfinite(Mr)&np.isfinite(Ms)
        C_cat = float(1.0 - np.nanmean(np.abs(Mr[mask]-Ms[mask])))

    # V (coverage)
    CC=[]
    for c in cat_cols:
        pr=real_cat[c].value_counts(normalize=True)
        ps=syn_cat[c].value_counts(normalize=True)
        cats=set(pr.index)
        cc=float(np.mean([k in ps.index for k in cats]))
        CC.append(cc)
    V_score = float(np.mean(CC)) if CC else np.nan

    # aggregate (weights like your cell)
    parts = []
    for x in [D_score, D_cat, C_num, C_cat, V_score]:
        if not np.isnan(x): parts.append(x)
    Q = float(np.mean(parts)) if parts else 0.0
    return Q

# ---------- P scorer (your DCR, Qδ, I) ----------
def score_privacy_P(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col="Target"):
    cols = [c for c in real_df.columns if c != target_col]
    cat_cols = [c for c in cols if real_df[c].dtype == "object"]
    num_cols = [c for c in cols if c not in cat_cols]

    def make_ohe():
        try: return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError: return OneHotEncoder(handle_unknown="ignore", sparse=False)

    enc = ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", make_ohe(),    cat_cols),
    ])
    X_real = enc.fit_transform(real_df[cols]); X_fake = enc.transform(syn_df[cols])

    # DCR
    nn = NearestNeighbors(n_neighbors=1).fit(X_real)
    dists,_ = nn.kneighbors(X_fake); dists=dists.ravel()
    DCR_mean = float(np.mean(dists))

    # Qδ
    quantiles = np.linspace(0.05,0.95,19); qd_vals=[]
    for c in num_cols:
        xr = np.asarray(real_df[c].dropna().values)
        xs = np.asarray(syn_df[c].dropna().values)
        if xr.size and xs.size:
            qr = np.quantile(xr, quantiles); qs = np.quantile(xs, quantiles)
            qd_vals.append(float(np.mean(np.abs(qr-qs))))
    Q_delta = float(np.mean(qd_vals)) if qd_vals else float("nan")

    # I
    def norm_df(df, cols):
        out=df[cols].copy()
        for c in cols:
            if np.issubdtype(out[c].dtype, np.number): out[c]=out[c].round(6)
            else: out[c]=out[c].astype(str).str.strip()
        return out
    real_norm = norm_df(real_df, cols); fake_norm = norm_df(syn_df, cols)
    dup_within_synth = float(1.0 - len(fake_norm.drop_duplicates())/max(1,len(fake_norm)))
    set_real=set(map(tuple, real_norm.to_numpy())); set_fake=set(map(tuple, fake_norm.to_numpy()))
    overlap_between = float(len(set_real & set_fake)/max(1,len(fake_norm)))

    # map to [0,1] scores (same idea as your cell)
    def normalize01(x, lo=0.0, hi=1.0):
        if np.isnan(x): return np.nan
        if hi==lo: return 0.0
        v=(x-lo)/(hi-lo); return float(max(0.0, min(1.0, v)))
    def inv01(x, cap=1.0):
        if np.isnan(x): return np.nan
        return float(max(0.0, min(1.0, 1.0 - x/cap)))

    d95 = float(np.percentile(dists, 95)) if len(dists) else 1.0
    DCR_score = normalize01(DCR_mean, lo=0.0, hi=d95 if d95>0 else 1.0)
    Qd_cap = float(np.nanmedian(qd_vals))*4 if qd_vals else (abs(Q_delta)+1e-6)
    Qdelta_score = inv01(Q_delta, cap=Qd_cap if Qd_cap>0 else 1.0)
    I_combined = 0.5*(dup_within_synth + overlap_between)
    I_score = inv01(I_combined, cap=0.10)

    P = 0.5*DCR_score + 0.3*Qdelta_score + 0.2*I_score
    return float(P)